# Compliance Gap Analysis — Demonstration

This notebook runs the project's compliance gap-finding system against five evaluation queries written in the voice of a Head of AI Compliance. For each question, the system returns a three-section gap analysis comparing what EU AI Act and UK GDPR require against what Novara TalentLens's policy documents say.

The system uses a single-call retrieval-augmented architecture:

1. **Retrieval** — BGE-large bi-encoder embeds the query, retrieves the top-5 most relevant law passages and the top-5 most relevant policy passages from a 1,140-chunk corpus.
2. **Generation** — a single LLM inference call (Gemma 2-2B) produces a three-section response: *what the law requires*, *what the policy says*, and *the gap*.
3. **Grounding** — a footer reports the retrieval-pool confidence (mean / max / min cosine) per side and a plain-English interpretation.

Empirical evaluation of the design choices (model size, prompt format, ranking strategy) is documented in `docs/evaluation-findings.md`. Per-decision rationale is in `docs/decisions.md`.

**Before running:** Runtime → Change runtime type → **GPU**.

## 1. Setup

In [ ]:
!git clone https://github.com/dariacoman/inst0100-compliance-gap-analysis.git
%cd inst0100-compliance-gap-analysis

In [ ]:
!pip install -q sentence-transformers transformers accelerate diskcache "numpy<2.2"

## 2. Authentication

The model used (`google/gemma-2-2b-it`) is gated on HuggingFace. Before running the next cell:

1. Accept the licence at [huggingface.co/google/gemma-2-2b-it](https://huggingface.co/google/gemma-2-2b-it)
2. Generate a read-token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Add it as a Colab secret named `HF_TOKEN` (left sidebar → key icon)

In [ ]:
import os
from google.colab import userdata

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded.')
except Exception:
    print('HF_TOKEN not found. Add it as a Colab secret to proceed.')

os.environ['PYTHONPATH'] = '/content/compliance-gap-analysis'

## 3. Verify environment

In [ ]:
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device = torch.cuda.get_device_properties(0)
    print(f'Device: {device.name} ({device.total_memory / 1e9:.1f} GB)')

In [ ]:
from src.simplified import analyse, LLM_MODEL_ID, EMBED_MODEL_ID

print(f'LLM:      {LLM_MODEL_ID}')
print(f'Embedder: {EMBED_MODEL_ID}')

## 4. Run a query

The cell below runs a single compliance question and returns a three-section gap analysis with a retrieval-grounding footer.

Five evaluation queries from `docs/test-queries.md` are pre-defined below as variables `Q1`–`Q5`. **Q5 (FRIA)** is the canonical demonstration query — it exercises the system's silence-detection on a deliberate gap.

| ID | Topic |
|---|---|
| Q1 | Multi-facet TalentLens compliance under EU AI Act + GDPR |
| Q2 | Red-teaming requirements before high-risk AI deployment |
| Q3 | GDPR Article 22 sub-clauses on solely automated decisions |
| Q4 | Transparency for candidates assessed by TalentLens |
| Q5 | Fundamental Rights Impact Assessment under EU AI Act Article 27 |

To run a different query, change `query = Q5` to `query = Q1` (or any other query string) and re-run the cell.

In [ ]:
Q1 = (
    "TalentLens compliance under EU AI Act Annex III \u00a74 \u2014 am I covered on "
    "Article 13 deployer instructions, Article 14 human oversight, Article 26 "
    "logs and worker information, and the related Article 22 GDPR "
    "automated-decisions duties? Where are the gaps?"
)
Q2 = (
    "Does our policy address the red-teaming requirements before deploying a "
    "high-risk AI system to production?"
)
Q3 = (
    "How do we meet GDPR Article 22 requirements on solely automated decisions "
    "affecting candidates \u2014 explicit consent, right to obtain human "
    "intervention, right to contest the decision, and right to express their "
    "point of view?"
)
Q4 = "Are we doing enough on transparency for candidates assessed by TalentLens?"
Q5 = (
    "Have we performed a Fundamental Rights Impact Assessment under EU AI Act "
    "Article 27 for TalentLens as a deployer of an Annex III high-risk system?"
)

query = Q5
print(analyse(query))

## 5. (Optional) Run all five queries

For reproducibility evidence, the cells below run all five evaluation queries in sequence and save the combined outputs to `colab_outputs.md`. Skip this section if you only need the live demonstration above.

In [ ]:
import time

QUERIES = {
    'Q1 multi-facet':           Q1,
    'Q2 red-teaming':           Q2,
    'Q3 Article 22 sub-clauses': Q3,
    'Q4 transparency':          Q4,
    'Q5 FRIA':                  Q5,
}

results = {}
for label, q in QUERIES.items():
    print(f'\n{"="*70}\n{label}\n{"="*70}')
    t0 = time.time()
    output = analyse(q)
    elapsed = time.time() - t0
    results[label] = {'query': q, 'output': output, 'elapsed_s': elapsed}
    print(output)
    print(f'\n[{elapsed:.1f}s]')

In [ ]:
from datetime import datetime, timezone

lines = [
    '# Compliance Gap Analysis \u2014 Colab Run',
    '',
    f'> Run date: {datetime.now(timezone.utc).strftime("%Y-%m-%d")}',
    f'> Model: `{LLM_MODEL_ID}`',
    f'> Embedder: `{EMBED_MODEL_ID}`',
    '',
    '---',
    '',
]

for label, r in results.items():
    lines += [
        f'## {label}', '',
        f'**Query:** {r["query"]}', '',
        f'**Latency:** {r["elapsed_s"]:.1f}s', '',
        '**Output:**', '',
        '```', r['output'].rstrip(), '```', '',
        '---', '',
    ]

with open('colab_outputs.md', 'w') as f:
    f.write('\n'.join(lines))

print('Saved to colab_outputs.md')